In [0]:
# Phase 3 : Comparison

crm_df = spark.table("workspace.default.crm_silver")

billing_df = spark.table("workspace.default.billing_silver")

analytics_df = spark.table("workspace.default.analytics_silver")

In [0]:
print("CRM Records:", crm_df.count())
print("Billing Records:", billing_df.count())
print("Analytics Records:", analytics_df.count())

CRM Records: 10195
Billing Records: 11454
Analytics Records: 894


In [0]:
crm_missing_in_billing = (
    crm_df.join(
        billing_df,
        on="customer_id",
        how="left_anti"
    )
)

print("Customers in CRM but missing in Billing:")

display(crm_missing_in_billing)

Customers in CRM but missing in Billing:


customer_id,name,email,signup_date,city
CRM000108,Riya Mehta,riya.mehta530@rediff.com,2022-06-17,Bangalore
CRM004992,Sneha Gupta,sneha.gupta883@gmail.com,2022-12-18,Mumbai
CRM004395,Kavya Nair,kavya.nair616@hotmail.com,2023-05-13,Kolkata
CRM008049,Diya Singh,diya.singh975@hotmail.com,2023-11-13,Delhi
CRM006240,Suresh Das,suresh.das120@yahoo.com,2023-11-12,Lucknow
CRM009272,Deepa Reddy,deepa.reddy415@outlook.com,2023-03-24,Chennai
CRM005857,Aarav Mishra,aarav.mishra688@hotmail.com,2023-01-25,Visakhapatnam
CRM009095,Aadhya Joshi,aadhya.joshi589@rediff.com,2024-01-11,Indore
CRM005404,Vivaan Shah,vivaan.shah261@yahoo.com,2022-01-03,Jaipur
CRM000583,Reyan Tiwari,reyan.tiwari768@hotmail.com,2022-03-21,Patna


In [0]:
print(
    "Missing Customers:",
    crm_missing_in_billing.count()
)

Missing Customers: 3969


In [0]:
billing_missing_in_crm = (
    billing_df.join(
        crm_df,
        on="customer_id",
        how="left_anti"
    )
)

print("Customers in Billing but missing in CRM")

display(billing_missing_in_crm)

Customers in Billing but missing in CRM


customer_id,transaction_id,amount,transaction_date,status
GHOST00880,TXN0011920,199.27,2023-04-08,refunded
GHOST00287,TXN0010924,247.97,2022-06-24,completed
GHOST00469,TXN0005520,211.82,2023-05-17,completed
CRM002347,TXN0001458,36.22,2023-05-09,failed
CRM005285,TXN0003561,69.29,2022-08-07,completed
GHOST00237,TXN0001442,224.3,2023-02-03,completed
GHOST00227,TXN0000645,78.06,2022-01-09,completed
GHOST00964,TXN0010303,435.9,2024-02-25,completed
GHOST00015,TXN0002925,3754.33,2024-06-22,pending
GHOST00853,TXN0007232,67.23,2024-04-11,completed


In [0]:
print(
    "Billing customers missing in CRM:",
    billing_missing_in_crm.count()
)

Billing customers missing in CRM: 1457


In [0]:
from pyspark.sql.functions import count

billing_summary = billing_df.groupBy("customer_id") \
    .agg(count("transaction_id").alias("transaction_count"))

display(billing_summary)

customer_id,transaction_count
CRM009140,2
CRM007473,2
CRM002988,1
GHOST00880,1
CRM004907,1
CRM007081,3
CRM002151,2
CRM005339,1
CRM008167,2
CRM009139,3


In [0]:
customer_transaction_summary = crm_df.join(
    billing_summary,
    on="customer_id",
    how="left"
)

display(customer_transaction_summary)

customer_id,name,email,signup_date,city,transaction_count
CRM000108,Riya Mehta,riya.mehta530@rediff.com,2022-06-17,Bangalore,null
CRM004992,Sneha Gupta,sneha.gupta883@gmail.com,2022-12-18,Mumbai,null
CRM002520,Vihaan Pillai,vihaan.pillai446@hotmail.com,2022-11-14,Vadodara,1
CRM004395,Kavya Nair,kavya.nair616@hotmail.com,2023-05-13,Kolkata,null
CRM000564,Riya Kumar,riya.kumar468@gmail.com,2022-05-14,Hyderabad,2
CRM009871,Vihaan Sharma,vihaan.sharma651@rediff.com,2022-03-10,Visakhapatnam,1
CRM008049,Diya Singh,diya.singh975@hotmail.com,2023-11-13,Delhi,null
CRM003350,Pooja Shah,pooja.shah584@outlook.com,2022-02-15,Coimbatore,1
CRM009939,Kabir Sinha,kabir.sinha330@yahoo.com,2022-08-12,Vadodara,2
CRM007145,Rohan Mehta,rohan.mehta565@yahoo.com,2023-11-21,Nagpur,1


In [0]:
from pyspark.sql.functions import col

customers_without_transactions = customer_transaction_summary.filter(
    col("transaction_count").isNull()
)

display(customers_without_transactions)

customer_id,name,email,signup_date,city,transaction_count
CRM000108,Riya Mehta,riya.mehta530@rediff.com,2022-06-17,Bangalore,null
CRM004992,Sneha Gupta,sneha.gupta883@gmail.com,2022-12-18,Mumbai,null
CRM004395,Kavya Nair,kavya.nair616@hotmail.com,2023-05-13,Kolkata,null
CRM008049,Diya Singh,diya.singh975@hotmail.com,2023-11-13,Delhi,null
CRM006240,Suresh Das,suresh.das120@yahoo.com,2023-11-12,Lucknow,null
CRM009272,Deepa Reddy,deepa.reddy415@outlook.com,2023-03-24,Chennai,null
CRM005857,Aarav Mishra,aarav.mishra688@hotmail.com,2023-01-25,Visakhapatnam,null
CRM009095,Aadhya Joshi,aadhya.joshi589@rediff.com,2024-01-11,Indore,null
CRM005404,Vivaan Shah,vivaan.shah261@yahoo.com,2022-01-03,Jaipur,null
CRM000583,Reyan Tiwari,reyan.tiwari768@hotmail.com,2022-03-21,Patna,null


In [0]:
print(
    "Customers without any transaction:",
    customers_without_transactions.count()
)

Customers without any transaction: 3911


In [0]:
from pyspark.sql.functions import sum

billing_revenue = billing_df.select(
    sum("amount")
).first()[0]

analytics_revenue = analytics_df.select(
    sum("total_revenue")
).first()[0]

print("Billing Revenue :", billing_revenue)
print("Analytics Revenue :", analytics_revenue)

print(
    "Difference:",
    abs(billing_revenue - analytics_revenue)
)

Billing Revenue : 6003866.280000019
Analytics Revenue : 4391369.719999994
Difference: 1612496.5600000247


In [0]:
comparison_report = spark.createDataFrame([
    ("CRM Records", crm_df.count()),
    ("Billing Records", billing_df.count()),
    ("Analytics Records", analytics_df.count()),
    ("CRM Missing in Billing", crm_missing_in_billing.count()),
    ("Billing Missing in CRM", billing_missing_in_crm.count()),
    ("Customers Without Transactions",
     customers_without_transactions.count())
], ["Metric", "Value"])

display(comparison_report)

Metric,Value
CRM Records,10195
Billing Records,11454
Analytics Records,894
CRM Missing in Billing,3911
Billing Missing in CRM,1457
Customers Without Transactions,3911


In [0]:
comparison_report.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.comparison_report")

In [0]:
display(
    spark.sql("""
    DESCRIBE HISTORY workspace.default.comparison_report
    """)
)

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-07-11T20:34:49.000Z,72969947334986,khandelwalvishwa1310@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1352019682178601),c50b5412-8966-444f-90aa-d61c1df80dd6,0711-201517-flo0f3me-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 6, numOutputBytes -> 1267)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
